# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

In [ ]:
import { init, Bool, Arith } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

## Auxiliary Functions

The symbolic transition system uses 3 variables.
* `M` is the number of missionaries on the left side of the river,
* `C` is the number of infidels on the left side of the river,
* `B` is the number of boats on the left side of the river.

We define these variables using `Z3.Int.const`.

In [ ]:
const M = Z3.Int.const('M'); const MX = Z3.Int.const('MX');
const C = Z3.Int.const('C'); const CX = Z3.Int.const('CX');
const B = Z3.Int.const('B'); const BX = Z3.Int.const('BX');

The function `start` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a `Z3` formula that specifies that everybody is on the western shore.
$$ \texttt{M} = 3 \wedge \texttt{C} = 3 \wedge \texttt{B} = 3 $$

In [ ]:
function start(M: Arith, C: Arith, B: Arith): Bool {
    return Z3.And(M.eq(3), C.eq(3), B.eq(1));
}

In [ ]:
start(M, C, B).toString()

The function `goal` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the eastern shore.
$$ \texttt{M} = 0 \wedge \texttt{C} = 0 \wedge \texttt{B} = 0 $$

In [ ]:
function goal(M: Arith, C: Arith, B: Arith): Bool {
    return Z3.And(M.eq(0), C.eq(0), B.eq(0));
}

The function `invariant` takes three `Z3` variables as input:
* `M` is the number of missionaries on the western shore,
* `C` is the number of infidels on the western shore,
* `B` is the number of boats on the western shore.

It returns an array of formulas. The conjunction of these formulas is `true` if there is no problem on either shore of the river.
There is no problem if either of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e. 
  $\texttt{M} = 0$.  
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of 
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of 
  missionaries and infidels have to match on the eastern shore as well.
* Furthermore, the number of missionaries and Cannibals is an element of the set $\{0,1,2,3\}$:
  $$ 0 \leq \texttt{M} \leq 3 \;\wedge\; 0 \leq \texttt{C} \leq 3$$

In [ ]:
function invariant(M: Arith, C: Arith, B: Arith): Bool[] {
    return [ Z3.Or(M.eq(0), M.eq(3), M.eq(C)),  // M = 0 ∨ M = 3 ∨ M = C
             M.ge(0), M.le(3),                  // 0 ≤ M ≤ 3 
             C.ge(0), C.le(3),                  // 0 ≤ C ≤ 3 
             B.ge(0), B.le(1)                   // 0 ≤ B ≤ 1 
           ];
}

In [ ]:
for (const constraint of invariant(M, C, B)) {
    console.log(constraint.toString());
}

The function `boatOK` takes 6 arguments:
* `Ma` is the number of missionaries on the eastern shore before the crossing.
* `Ca` is the number of infidels on the eastern shore before the crossing.
* `Ba` is the number of boats on the eastern shore before the crossing. 
* `Mb` is the number of missionaries on the eastern shore after the crossing.
* `Cb` is the number of infidels on the eastern shore after the crossing.
* `Bb` is the number of infidels on the eastern shore after the crossing.

The function returns a formula that checks that if the boat 
crosses the river from the western shore to the eastern shore, then the number of people on the boat is between $1$ and $2$
and the number of missionaries and cannibals on the western shore can't increase:
$$ \mathtt{Ba} = 1 \;\rightarrow\; 1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2 \,\wedge\, \mathtt{Mb} \leq \mathtt{Ma} \,\wedge\, \mathtt{Cb} \leq \mathtt{Ca}$$
Here `Ma - Mb` is the number of missionaries on the boat and  `Ca - Cb` is the number of infidels on the boat.  The formula 
$1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2$ expresses that there is at least one but not more than two passengers on the boat.

In [ ]:
function boatOK(Ma: Arith, Ca: Arith, Ba: Arith, Mb: Arith, Cb: Arith, Bb: Arith): Bool {
    const numMis = Ma.sub(Mb);           // Ma - Mb
    const numCan = Ca.sub(Cb);           // Ca - Cb
    const all    = numMis.add(numCan);   // Ma - Mb + Ma - Mb
    // Ba = 1 → 1 ≤ Ma - Mb + Ma - Mb ≤ 2 ∧ Mb ≤ Ma ∧ Cb ≤ Ca
    return Z3.Implies(Ba.eq(1), 
                      Z3.And(all.ge(1), all.le(2), Mb.le(Ma), Cb.le(Ca)));
}

In [ ]:
boatOK(M, C, B, MX, CX, BX).toString()

The function `transition` takes 6 arguments:
* `Ma` is the number of missionaries on the eastern shore before the crossing.
* `Ca` is the number of infidels on the eastern shore before the crossing.
* `Ba` is the number of boats on the eastern shore before the crossing. 
* `Mb` is the number of missionaries on the eastern shore after the crossing.
* `Cb` is the number of infidels on the eastern shore after the crossing.
* `Bb` is the number of infidels on the eastern shore after the crossing.

It returns a formula that expresses the following:
* The boat changes the sides in every transition:
  $$ \mathtt{Bb} = 1 - \mathtt{Ba} $$
* If the boat crosses the river from the western shore to the eastern shore,
  then there is at least one and at most two passengers and the numbers on the western shore do not increase.
* If the boat crosses the river from the eastern shore to the western shore,
  then there is at least one and at most two passengers and the numbers on the eastern shore do not increase.

In [ ]:
function transition(Ma: Arith, Ca: Arith, Ba: Arith, Mb: Arith, Cb: Arith, Bb: Arith): Bool[] { 
    const formulas = [ Bb.eq(Ba.mul(-1).add(1)) ]; // Bb = 1 - Ba
    formulas.push(boatOK(Ma, Ca, Ba, Mb, Cb, Bb));
    formulas.push(boatOK(Mb, Cb, Bb, Ma, Ca, Ba));
    return formulas;
}

In [ ]:
transition(M, C, B, MX, CX, BX).toString();

In [ ]:
import { RecursiveMap as Map } from "recursive-set";

The async function `missionariesCSP` creates a CSP that tries to solve the problem with `n` crossings. It creates a new `Solver` instance, adds all constraints (start, goal, invariants, transitions) and checks for satisfiability.

In [ ]:
async function missionariesCSP(n: number): Promise<Map<string, number>> {
    const solver = new Z3.Solver();
    const Ms = [], Cs = [], Bs = [];
    for (let i = 0; i <= n; ++i) {
        Ms.push(Z3.Int.const(`M${i}`));
        Cs.push(Z3.Int.const(`C${i}`));
        Bs.push(Z3.Int.const(`B${i}`));
    }
    solver.add(start(Ms[0], Cs[0], Bs[0]));
    solver.add(goal (Ms[n], Cs[n], Bs[n]));
    for (let i = 0; i < n; ++i) {
        solver.add(...invariant (Ms[i], Cs[i], Bs[i]));
        solver.add(...transition(Ms[i], Cs[i], Bs[i], Ms[i+1], Cs[i+1], Bs[i+1]));
    }
    const result = await solver.check();
    if (result == 'sat') {
        const model = solver.model();
        const solution = new Map<string, number>();
        for (let i = 0; i <= n; i++) {
            solution.set(`M${i}`, parseInt(model.eval(Ms[i]).toString()));
            solution.set(`C${i}`, parseInt(model.eval(Cs[i]).toString()));
            solution.set(`B${i}`, parseInt(model.eval(Bs[i]).toString()));
        }
        return solution;
    } 
}

The function `findSolution` computes a solution to the problem of saving the infidels.

In [ ]:
async function findSolution(): Promise<Map<string, number>> {
    let n = 1;
    while (true) {
        console.log(`Trying n=${n}...`);
        const solution = await missionariesCSP(n);
        if (solution) {
            return solution;
        }
        n += 2;
    }
}

On my desktop computer (2017 iMac with 3.4 GHz Quad-Core Intel i5) it takes about 2 seconds to solve the problem. 

In [ ]:
console.time("Missionaries Solving");
const solution = await findSolution();
console.timeEnd("Missionaries Solving");
solution

In [ ]:
function show_solution(sol: Map<string, number>) {
    const n = sol.size / 3 - 1;
    if (!sol) return;
    for (let i = 0; i <= n; i++) {
        const M = sol.get(`M${i}`)!;
        const C = sol.get(`C${i}`)!;
        const B = sol.get(`B${i}`)!;
        const west = '😇'.repeat(M) + '🥷'.repeat(C);
        const east = '😇'.repeat(3 - M) + '🥷'.repeat(3 - C);
        const riverWidth = 28; 
        let line = `${west}${' '.repeat(riverWidth)}${east}`;
        console.log(line);
        if (B === 1 && i < n) {
            const MB = sol.get(`M${i}`)! - sol.get(`M${i+1}`)!;
            const CB = sol.get(`C${i}`)! - sol.get(`C${i+1}`)!;
            const boatContent = '😇'.repeat(MB) + '🥷'.repeat(CB);
            console.log(' '.repeat(12) + '>>> ' + boatContent + ' >>>');
        } else if (i < n) {
            const MB = sol.get(`M${i+1}`)! - sol.get(`M${i}`)!;
            const CB = sol.get(`C${i+1}`)! - sol.get(`C${i}`)!;
            const boatContent = '😇'.repeat(MB) + '🥷'.repeat(CB);
            console.log(' '.repeat(12) + '<<< ' + boatContent + ' <<<');
        }
    }
}

In [ ]:
show_solution(solution);

In [ ]:
import * as tslab from "tslab";

function show_animated_solution(sol: Map<string, number>) {
    const n = sol.size / 3 - 1;
    if (!sol) {
        console.error("No solution provided.");
        return;
    }

    // Prepare the state data for the frontend JavaScript
    const states = [];
    for (let i = 0; i <= n; i++) {
        states.push({ M: sol.get(`M${i}`), C: sol.get(`C${i}`), B: sol.get(`B${i}`) });
    }

    // Generate a unique ID suffix to avoid element collisions if run multiple times
    const uid = Math.random().toString(36).substring(2, 9);

    const htmlContent = `
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-${uid}" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-${uid}" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-${uid}" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-${uid}" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of ${n}</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-m-${uid}" x="75" y="100" font-size="28" text-anchor="middle">😇😇😇</text>
            <text id="west-c-${uid}" x="75" y="150" font-size="28" text-anchor="middle">🥷🥷🥷</text>
            
            <text id="east-m-${uid}" x="525" y="100" font-size="28" text-anchor="middle"></text>
            <text id="east-c-${uid}" x="525" y="150" font-size="28" text-anchor="middle"></text>

            <g id="boat-${uid}" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-${uid}" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = ${JSON.stringify(states)};
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-${uid}');
        const westM = document.getElementById('west-m-${uid}');
        const westC = document.getElementById('west-c-${uid}');
        const eastM = document.getElementById('east-m-${uid}');
        const eastC = document.getElementById('east-c-${uid}');
        const boatP = document.getElementById('boat-p-${uid}');
        const stepLabel = document.getElementById('label-step-${uid}');
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            // DIESEL ENGINE SIMULATION:
            // A sawtooth wave at ~12 Hz creates a rapid "clicking/knocking" 
            // sound that mimics the slow chug of a diesel engine.
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            
            // Low-pass filter to muffle the harsh highs and keep it sounding bassy/heavy
            filter.type = 'lowpass';
            filter.frequency.value = 150;

            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            // Using a slightly higher gain (0.6) because extreme low frequencies are perceived as quieter
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4); // Slower fade out for realism
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            westM.textContent = '😇'.repeat(s.M);
            westC.textContent = '🥷'.repeat(s.C);
            eastM.textContent = '😇'.repeat(3 - s.M);
            eastC.textContent = '🥷'.repeat(3 - s.C);
            boatP.textContent = '';
            
            boat.style.transform = s.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of ${n}';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const mPass = Math.abs(s0.M - s1.M);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.B === 1) { 
                westM.textContent = '😇'.repeat(s0.M - mPass);
                westC.textContent = '🥷'.repeat(s0.C - cPass);
            } else { 
                eastM.textContent = '😇'.repeat((3 - s0.M) - mPass);
                eastC.textContent = '🥷'.repeat((3 - s0.C) - cPass);
            }
            boatP.textContent = '😇'.repeat(mPass) + '🥷'.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-${uid}').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-${uid}').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-${uid}').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    `;

    tslab.display.html(htmlContent);
}

In [ ]:
show_animated_solution(solution)